# ***Step-1-Extract-data-from-telegram-channel***

## ***Import libs***

In [ ]:
import requests
import pandas as pd
import numpy as np

import re
import time

from bs4 import BeautifulSoup


## ***Extract images***

In [ ]:


CHANNEL = "citysilverandbags925"
BASE_URL = f"https://t.me/s/{CHANNEL}"

MAX_PAGES = 1000

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

all_messages = {}
previous_oldest_id = None

for page in range(1, MAX_PAGES + 1):

    if previous_oldest_id is None:
        url = BASE_URL
    else:
        url = f"{BASE_URL}?before={previous_oldest_id}"

    print(f"\nPage {page}: {url}")

    response = session.get(url, timeout=30)

    if response.status_code == 429:
        print("Telegram rate limit reached. Stopping safely.")
        break

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    messages = soup.select(".tgme_widget_message")

    if not messages:
        print("No messages returned. Reached available preview boundary.")
        break

    page_ids = []
    new_messages = 0

    for message in messages:

        post_id = message.get("data-post")

        if not post_id:
            continue

        message_id = int(post_id.split("/")[-1])
        page_ids.append(message_id)

        if post_id in all_messages:
            continue

        new_messages += 1

        text_element = message.select_one(
            ".tgme_widget_message_text"
        )

        caption = (
            text_element.get_text(" ", strip=True)
            if text_element else ""
        )

        image_urls = []

        # Collect all photo elements in this post
        for photo in message.select(
            ".tgme_widget_message_photo_wrap"
        ):

            style = photo.get("style", "")

            urls = re.findall(
                r"""url\(['"]?(.*?)['"]?\)""",
                style
            )

            image_urls.extend(urls)

        # Remove duplicate image URLs
        image_urls = list(dict.fromkeys(image_urls))

        all_messages[post_id] = {
            "message_id": message_id,
            "post_url": f"https://t.me/{CHANNEL}/{message_id}",
            "caption": caption,
            "image_urls": image_urls,
            "image_count": len(image_urls)
        }

    if not page_ids:
        print("No valid message IDs. Stopping.")
        break

    oldest_id = min(page_ids)

    print("Messages returned:", len(page_ids))
    print("New messages collected:", new_messages)
    print("Oldest ID:", oldest_id)
    print("Total unique messages:", len(all_messages))

    # Check whether pagination is moving backwards
    if previous_oldest_id is not None:
        if oldest_id >= previous_oldest_id:
            print("Pagination did not move to older messages.")
            break

    if new_messages == 0:
        print("No new messages collected. Stopping.")
        break

    previous_oldest_id = oldest_id

    time.sleep(2)

# Export all collected messages
messages_df = pd.DataFrame(all_messages.values())

# Export one row per image
image_rows = []

for post in all_messages.values():

    for image_index, image_url in enumerate(
        post["image_urls"], start=1
    ):

        image_rows.append({
            "message_id": post["message_id"],
            "post_url": post["post_url"],
            "caption": post["caption"],
            "image_number": image_index,
            "image_url": image_url
        })

images_df = pd.DataFrame(image_rows)

images_df.to_csv(
    "../data/telegram_images.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== COLLECTION REPORT ==========")
print("Pages fetched:", page)
print("Unique messages:", len(messages_df))
print("Messages with images:",
      (messages_df["image_count"] > 0).sum())
print("Total extracted images:", len(images_df))

if len(messages_df):
    print("Newest message ID:", messages_df["message_id"].max())
    print("Oldest message ID:", messages_df["message_id"].min())

print("\nFiles saved:")
print("- telegram_messages.csv")
print("- telegram_messages.json")
print("- telegram_images.csv")



Page 1: https://t.me/s/citysilverandbags925
Messages returned: 5
New messages collected: 5
Oldest ID: 1427
Total unique messages: 5

Page 2: https://t.me/s/citysilverandbags925?before=1427
Messages returned: 7
New messages collected: 7
Oldest ID: 1387
Total unique messages: 12

Page 3: https://t.me/s/citysilverandbags925?before=1387
Messages returned: 6
New messages collected: 6
Oldest ID: 1365
Total unique messages: 18

Page 4: https://t.me/s/citysilverandbags925?before=1365
Messages returned: 8
New messages collected: 8
Oldest ID: 1341
Total unique messages: 26

Page 5: https://t.me/s/citysilverandbags925?before=1341
Messages returned: 15
New messages collected: 15
Oldest ID: 1321
Total unique messages: 41

Page 6: https://t.me/s/citysilverandbags925?before=1321
Messages returned: 13
New messages collected: 13
Oldest ID: 1301
Total unique messages: 54

Page 7: https://t.me/s/citysilverandbags925?before=1301
Messages returned: 2
New messages collected: 2
Oldest ID: 1281
Total unique 

### ***Show images***

In [ ]:
images = pd.read_csv('../data/telegram_images.csv')

In [11]:
images.head()

,message_id,post_url,caption,image_number,image_url
0,1427,https://t.me/citysilverandbags925/1427,NaN,1,https://cdn4.telesco.pe/file/rJe9lCd8RmGttUAiR...
1,1427,https://t.me/citysilverandbags925/1427,NaN,2,https://cdn4.telesco.pe/file/R0F5LtF3FUTNDQq_9...
2,1427,https://t.me/citysilverandbags925/1427,NaN,3,https://cdn4.telesco.pe/file/ehAXJjuNVTccu9a_v...
3,1427,https://t.me/citysilverandbags925/1427,NaN,4,https://cdn4.telesco.pe/file/Zn8AafSrVBwrdM-os...
4,1427,https://t.me/citysilverandbags925/1427,NaN,5,https://cdn4.telesco.pe/file/BYVw1PdWYbCI0tmqd...


### ***Add name and desc***

### ***Add image***

### ***Add category***

In [12]:

PRODUCT_CATEGORIES = {
    "Rings": ["Female", "Male"],
    "Bracelets": ["Female", "Male"],
    "Necklaces": ["Female", "Male"],
    "Earrings": ["Female"],
    "Accessories": ["Unisex"],
    "Sets": ["Female", "Male"],
    "Bags": ["Female"]
}


### ***Add colors***

### ***Add types***

### ***Add sizes***

### ***Add gm***

### ***Add price***